# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading and exploring a Croissant-format dataset using the `mlcroissant` library. All dataset elements—record sets, fields, and columns—are referenced by their `@id`.

### Dataset Source
The dataset source is a Croissant schema published at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Dataset object
dataset = mlc.Dataset(croissant_url)

# Access core metadata (name & description)
meta_json = dataset.metadata.to_json()
print(f"Dataset Name: {meta_json['name']}")
print(f"Description: {meta_json['description']}")

## 2. Data Overview
List all available record sets, fields, and their `@id`. This information is crucial for referencing and extraction in later steps.

In [ ]:
# List all record sets and fields, referencing only their @id

print("Available Record Sets (by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    * Field @id: {field.id}, name: {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` from the above overview.

In [ ]:
# Identify available record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load all records for each record set into dataframes
for recset_id in record_set_ids:
    print(f"Loading records for RecordSet {recset_id}...")
    records_iter = dataset.records(record_set=recset_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"→ Columns: {df.columns.tolist()}")
        print(df.head(1))
    else:
        print(f"→ No records found.")

# For demo, select the first loaded record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"Using RecordSet: {main_record_set_id}")
    print(f"Columns: {main_df.columns.tolist()}")
    display(main_df.head())
else:
    print("No datasets with records were found.")

## 4. Exploratory Data Analysis (EDA)
Let's analyze numeric fields: for demonstration, choose the first (or a relevant) numeric column by its field `@id` for processing.

In [ ]:
# Find a numeric field in the RecordSet for analysis
from pandas.api.types import is_numeric_dtype

numeric_fields = [col for col in main_df.columns if is_numeric_dtype(main_df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # select first numeric column
    print(f"Example numeric field (@id): {numeric_field_id}")
else:
    print("No numeric fields available for EDA.")

# Threshold for demonstration
threshold = main_df[numeric_field_id].quantile(0.75) if numeric_fields else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold] if numeric_fields else main_df
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

if numeric_fields:
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

# Attempt grouping by another non-numeric field if available
group_fields = [col for col in main_df.columns if main_df[col].dtype == object and col != numeric_field_id]
if group_fields:
    group_field_id = group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped averages by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the numeric field and group summaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(6, 4))
    sns.histplot(main_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_fields:
        plt.figure(figsize=(8, 4))
        order = grouped_df.sort_values(numeric_field_id)[group_field_id] if 'grouped_df' in locals() else []
        sns.barplot(
            data=grouped_df,
            x=group_field_id,
            y=numeric_field_id,
            order=order
        )
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and performing basic analyses on the Croissant FAIR^2 dataset `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` using the `mlcroissant` library. Record sets, fields, and all key data assets were referenced by their `@id` as required by the FAIR Croissant standard.

You can further extend this notebook to perform advanced analyses, train machine learning models, or propagate updates as new versions of the dataset are published.